# 03 Modeling

This notebook trains candidate classification models to predict whether an employee is likely to leave. It compares an interpretable baseline with tree-based models and documents a leakage-aware feature strategy.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 50)

In [ ]:
from evaluate import compare_models, get_feature_importance, plot_feature_importance
from feature_engineering import add_retention_features
from train import prepare_train_test_data, train_candidate_models
from utils import DATA_PROCESSED, load_processed_data, save_plot

clean_df = load_processed_data()
df = add_retention_features(clean_df)

## Modeling Strategy

The target is binary: `left = 1` if the employee left and `left = 0` if the employee stayed.

Models compared:

- Logistic Regression: interpretable baseline
- Decision Tree: interpretable nonlinear model
- Random Forest: stronger ensemble model for final evaluation

The selected portfolio model uses a leakage-aware feature set that removes `satisfaction_level` and replaces raw `average_monthly_hours` with an `overworked` indicator. This keeps the project closer to a practical HR early-intervention workflow.

In [ ]:
X_train, X_test, y_train, y_test = prepare_train_test_data(df, leakage_aware=True)
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True).round(3))

In [ ]:
models = train_candidate_models(X_train, y_train)
model_comparison = compare_models(models, X_test, y_test)
model_comparison

In [ ]:
comparison_path = DATA_PROCESSED / "model_comparison.csv"
model_comparison.to_csv(comparison_path, index=False)
comparison_path

## Feature Importance

For the final Random Forest model, feature importance helps translate predictions into business levers. The model should support HR conversations about workload and promotion processes rather than automate employee decisions.

In [ ]:
rf_model = models["Random Forest"]
importance = get_feature_importance(rf_model)
importance.head(12)

In [ ]:
fig, ax = plot_feature_importance(importance, top_n=12)
save_plot(fig, "images/modeling/random_forest_feature_importance.png")
plt.show()

## Modeling Takeaways

- Recall is prioritized because missing an at-risk employee is costly for retention planning.
- Precision remains important because HR outreach should be focused and respectful.
- Random Forest is the preferred candidate because it captures nonlinear attrition patterns while still allowing feature importance review.